# MedNorm-VI S1 - Internal-Test Evaluation of the Full-Training Best Checkpoint

**READ-ONLY EVALUATION. Colab GPU runtime. No training, no organizer inference, no `output.zip`.**

Loads `checkpoints/best.pt` from a **validated** full-training artifact and scores it on the
governed `internal_test` split.

## What this notebook does NOT do

* it never trains, fine-tunes, or takes an optimizer step;
* it never writes into the full-training artifact directory - results go to a **separate**
  evaluation directory;
* it never runs organizer inference and never produces `output.zip`;
* it never reads the organizer's private test data.

## TWO-PASS EXECUTION

`Runtime > Run all`, wait for the forced kernel restart, then `Run all` **again**.

## RUNTIME INPUTS

| Input | Meaning |
| --- | --- |
| `FULL_TRAINING_ARTIFACT_DIR` | the completed run to evaluate |
| `EXPECTED_BEST_CHECKPOINT_SHA256` | 64-hex digest of the `best.pt` you are accepting |
| `EXPECTED_LATEST_CHECKPOINT_SHA256` | 64-hex digest of `latest.pt` |
| `PINNED_MODEL_REVISION` | the immutable revision the run was trained on |

The digests default to **empty** and are never hardcoded in source, so accepting a run needs no
code edit. Validation runs first and evaluation is refused unless it passes.

## 1. Configuration (no scientific imports)

In [ ]:
from __future__ import annotations

import importlib
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/MedNorm-VI")
REPO_DIR = Path("/content/MedNorm-VI")
REPO_URL = "https://github.com/vquclinh/MedNorm-VI.git"
REPO_REF = "main"
for _name, _var in (("MEDNORM_DRIVE_ROOT", "DRIVE_ROOT"), ("MEDNORM_REPO_DIR", "REPO_DIR")):
    if _name in os.environ:
        globals()[_var] = Path(os.environ[_name])
if "MEDNORM_REPO_URL" in os.environ:
    REPO_URL = os.environ["MEDNORM_REPO_URL"]
if "MEDNORM_REPO_REF" in os.environ:
    REPO_REF = os.environ["MEDNORM_REPO_REF"]

CORPUS_DIR = DRIVE_ROOT / "data" / "derived" / "training_corpora" / "mednorm_vi_training_v1"
MODEL_CACHE_DIR = DRIVE_ROOT / "model_cache" / "huggingface"

# ---------------------------------------------------------------------------
# RUNTIME INPUTS - the completed run to evaluate. Nothing here is hardcoded.
# ---------------------------------------------------------------------------
FULL_TRAINING_ARTIFACT_DIR = Path(os.environ.get(
    "MEDNORM_FULL_TRAINING_ARTIFACT_DIR",
    str(DRIVE_ROOT / "artifacts" / "s1_mention_full_training_v1")))
EXPECTED_BEST_CHECKPOINT_SHA256 = os.environ.get(
    "MEDNORM_EXPECTED_BEST_CHECKPOINT_SHA256", "")
EXPECTED_LATEST_CHECKPOINT_SHA256 = os.environ.get(
    "MEDNORM_EXPECTED_LATEST_CHECKPOINT_SHA256", "")
PINNED_MODEL_REVISION = os.environ.get("MEDNORM_PINNED_MODEL_REVISION", "")
EXPECTED_COMPLETED_EPOCHS = int(os.environ.get("MEDNORM_EXPECTED_EPOCHS", "4"))
EXPECTED_COMPLETED_OPTIMIZER_STEPS = int(
    os.environ.get("MEDNORM_EXPECTED_OPTIMIZER_STEPS", "2976"))

# Results NEVER go into the training artifact; that directory is read-only here.
EVAL_SPLIT = "internal_test"

IN_COLAB_BOOTSTRAP = "google.colab" in sys.modules
# LOCAL mode uses the git-ignored local checkpoint and the best-checkpoint-only
# gate; COLAB mode uses the complete Drive artifact and the full-artifact gate.
EXECUTION_MODE = "colab" if IN_COLAB_BOOTSTRAP else "local"
# Colab writes beside the Drive artifacts; local writes under the ignored reports/.
EVAL_OUTPUT_DIR = (
    DRIVE_ROOT / "artifacts" / "s1_mention_internal_test_eval_v1"
    if IN_COLAB_BOOTSTRAP else Path("reports") / "s1_internal_test_eval")
print(json.dumps({
    "mode": "READ_ONLY_EVALUATION",
    "full_training_artifact_dir": str(FULL_TRAINING_ARTIFACT_DIR),
    "eval_output_dir": str(EVAL_OUTPUT_DIR),
    "eval_split": EVAL_SPLIT,
    "execution_mode": EXECUTION_MODE,
    "expected_best_sha256_supplied": bool(EXPECTED_BEST_CHECKPOINT_SHA256),
    "pinned_revision_supplied": bool(PINNED_MODEL_REVISION),
}, indent=2, sort_keys=True))

## 2. Repository checkout (stdlib only)

In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run([
    "git",
    "clone",
    "--branch",
    REPO_REF,
    "--single-branch",
    REPO_URL,
    str(REPO_DIR),
], check=True)
RESOLVED_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert len(RESOLVED_COMMIT) == 40 and all(c in "0123456789abcdef" for c in RESOLVED_COMMIT)
assert (REPO_DIR / "src" / "mednorm_vi").is_dir(), "cloned repo is missing src/mednorm_vi"
sys.path.insert(0, str(REPO_DIR / "src"))
print(json.dumps({"resolved_commit": RESOLVED_COMMIT, "src_verified": True}, indent=2, sort_keys=True))


## 3. Dependency metadata inspection

In [ ]:
import importlib.metadata as importlib_metadata

sys.path.insert(0, str(REPO_DIR / "src"))
from mednorm_vi.training.colab_bootstrap import (  # noqa: E402
    INSTALL_AND_RESTART,
    PROCEED,
    build_install_command,
    build_marker_fingerprint,
    build_pip_constraints,
    classify_dependency_health,
    compute_dependency_closure,
    decide_bootstrap_action,
    load_dependency_contract,
    marker_mismatches,
    normalize_distribution_name,
    validate_abi_report,
    validate_install_command,
)

DEPENDENCY_CONTRACT_PATH = REPO_DIR / "configs" / "training" / "s1_mention_colab_dependencies.yaml"
contract = load_dependency_contract(DEPENDENCY_CONTRACT_PATH)
DEPENDENCY_CONTRACT_VERSION = contract.contract_version

TRACKED_PACKAGES = (
    "numpy", "torch", "torchvision", "torchaudio", "transformers", "tokenizers",
    "huggingface_hub", "sentencepiece", "accelerate", "py_vncorenlp", "scipy",
    "pandas", "scikit-learn", "safetensors", "pyarrow",
)

def package_version(name):
    """Version via metadata only - never imports the package (no NumPy load)."""
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return ""

baseline_versions = {name: package_version(name) for name in TRACKED_PACKAGES}
assert baseline_versions["numpy"], "no baseline NumPy detected in the Colab image"
assert baseline_versions["torch"], "no baseline Torch detected in the Colab image"

# The marker fingerprint binds a skipped installation to THIS tracked contract
# (exact file bytes), THIS Python major.minor, THIS protected baseline, and THIS
# normalized requirement set. A version string alone is far too weak.
PYTHON_MAJOR_MINOR = f"{sys.version_info.major}.{sys.version_info.minor}"
MARKER_FINGERPRINT = build_marker_fingerprint(
    contract, PYTHON_MAJOR_MINOR, baseline_versions)
DEPENDENCY_CONTRACT_SHA256 = contract.contract_sha256
INSTALL_REQUIREMENT_HASH = contract.install_requirement_hash
print(json.dumps({
    "baseline_versions": {k: v for k, v in baseline_versions.items() if v},
    "marker_fingerprint": MARKER_FINGERPRINT.as_dict(),
}, indent=2, sort_keys=True))


## 4. Consolidated installation + forced kernel restart (PASS 1 only)

In [ ]:
MARKER_PATH = Path(contract.marker_path)
CONSTRAINT_PATH = Path("/content/mednorm_s1_constraints.txt")

bootstrap_marker = None
if MARKER_PATH.is_file():
    try:
        bootstrap_marker = json.loads(MARKER_PATH.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        bootstrap_marker = None

# Every fingerprint field must match before installation may be skipped.
BOOTSTRAP_MARKER_MISMATCHES = marker_mismatches(bootstrap_marker, MARKER_FINGERPRINT)
BOOTSTRAP_ACTION = decide_bootstrap_action(bootstrap_marker, MARKER_FINGERPRINT)
print(json.dumps({
    "bootstrap_action": BOOTSTRAP_ACTION,
    "marker_path": str(MARKER_PATH),
    "marker_mismatches": BOOTSTRAP_MARKER_MISMATCHES,
    "expected_fingerprint": MARKER_FINGERPRINT.as_dict(),
}, indent=2, sort_keys=True))

if BOOTSTRAP_ACTION == INSTALL_AND_RESTART:
    # Pin the INHERITED stack to the versions this runtime already provides so pip
    # cannot silently move NumPy/Torch. An incompatible request now fails loudly
    # instead of corrupting the C-ABI.
    constraints = build_pip_constraints(baseline_versions)
    CONSTRAINT_PATH.write_text("\n".join(constraints) + "\n", encoding="utf-8")
    install_command = build_install_command(contract, str(CONSTRAINT_PATH), sys.executable)
    validate_install_command(install_command)
    print("constraints:", constraints)
    print("install:", " ".join(install_command))
    subprocess.run(install_command, check=True)
    pip_check = subprocess.run(
        [sys.executable, "-m", "pip", "check"], capture_output=True, text=True, check=False)
    pip_check_output = "\n".join(
        part for part in (pip_check.stdout.strip(), pip_check.stderr.strip()) if part)
    # Written with the CURRENT fingerprint, so PASS 2 matches exactly and the
    # notebook can never restart twice for the same environment.
    MARKER_PATH.write_text(json.dumps({
        "install_completed": True,
        **MARKER_FINGERPRINT.as_dict(),
        "baseline_versions": baseline_versions,
        "constraints": constraints,
        "installed_specifiers": list(contract.install_specifiers),
        "pip_check_returncode": pip_check.returncode,
        "pip_check_stdout": pip_check.stdout,
        "pip_check_stderr": pip_check.stderr,
        "pip_check_output": pip_check_output,
    }, indent=2, sort_keys=True), encoding="utf-8")
    print("=" * 78)
    print("PASS 1 COMPLETE - the kernel is about to restart (this is expected).")
    print("AFTER the restart finishes, run Runtime > Run all AGAIN to execute PASS 2.")
    print("=" * 78)
    if IN_COLAB_BOOTSTRAP:
        os.kill(os.getpid(), 9)  # real kernel restart; Colab reconnects automatically
    else:
        raise SystemExit("dependency installation requires a kernel restart")
else:
    print("PASS 2: bootstrap marker matches every fingerprint field;",
          DEPENDENCY_CONTRACT_VERSION, DEPENDENCY_CONTRACT_SHA256[:16])

DEPENDENCY_RESTART_COMPLETED = BOOTSTRAP_ACTION == PROCEED


## 5. Post-restart dependency verification (scoped S1 health)

In [ ]:
assert DEPENDENCY_RESTART_COMPLETED, (
    "PASS 1 installs dependencies and restarts the kernel. Run all cells again for PASS 2.")

# Re-validate the marker in the restarted kernel: PASS 1 wrote it in a different
# process, so the fingerprint is re-checked here against the live runtime.
POST_RESTART_MARKER_MISMATCHES = marker_mismatches(bootstrap_marker, MARKER_FINGERPRINT)
assert not POST_RESTART_MARKER_MISMATCHES, (
    f"bootstrap marker no longer matches this runtime: {POST_RESTART_MARKER_MISMATCHES}")

installed_versions = {name: package_version(name) for name in TRACKED_PACKAGES}
marker_baseline = dict(bootstrap_marker.get("baseline_versions", {}))
changed_packages = {
    name: {"baseline": marker_baseline.get(name, ""), "current": installed_versions[name]}
    for name in TRACKED_PACKAGES
    if marker_baseline.get(name, "") != installed_versions[name]
}
protected_changed = {
    name: change for name, change in changed_packages.items()
    if name in ("numpy", "torch", "torchvision", "torchaudio")
}
assert not protected_changed, (
    f"inherited stack was modified despite constraints: {protected_changed}")
# `pip check` is captured IN FULL (stdout and stderr, never truncated) and kept as a
# diagnostic. It audits the whole Colab image, so its global verdict alone must not
# gate S1: preinstalled Gradio/IPython complaints are unrelated to this smoke.
pip_check_proc = subprocess.run(
    [sys.executable, "-m", "pip", "check"], capture_output=True, text=True, check=False)
PIP_CHECK_OUTPUT = "\n".join(
    part for part in (pip_check_proc.stdout.strip(), pip_check_proc.stderr.strip()) if part)
PIP_CHECK_PASSED = pip_check_proc.returncode == 0

def installed_requirement_graph():
    """Requirement graph from metadata only - imports nothing (no NumPy load)."""
    graph = {}
    for dist in importlib_metadata.distributions():
        dist_name = normalize_distribution_name(dist.metadata["Name"] or "")
        if dist_name:
            graph.setdefault(dist_name, []).extend(dist.requires or [])
    return graph

# The closure is what S1 ACTUALLY depends on: the contract's import roots plus their
# transitive requirements, resolved from the installed metadata of this runtime.
S1_DEPENDENCY_CLOSURE = compute_dependency_closure(
    contract.closure_root_distributions, installed_requirement_graph())
DEPENDENCY_HEALTH = classify_dependency_health(
    PIP_CHECK_OUTPUT, S1_DEPENDENCY_CLOSURE, pip_check_proc.returncode)
print("pip check output (complete):")
print(PIP_CHECK_OUTPUT or "(no broken requirements reported)")
print(json.dumps({
    "bootstrap_action": BOOTSTRAP_ACTION,
    "marker_mismatches": POST_RESTART_MARKER_MISMATCHES,
    "dependency_contract_sha256": DEPENDENCY_CONTRACT_SHA256,
    "install_requirement_hash": INSTALL_REQUIREMENT_HASH,
    "python_major_minor": PYTHON_MAJOR_MINOR,
    "changed_packages": changed_packages,
    "protected_stack_unchanged": True,
    "pip_check_passed": PIP_CHECK_PASSED,
    "s1_dependency_closure_size": len(S1_DEPENDENCY_CLOSURE),
    "s1_dependency_healthy": DEPENDENCY_HEALTH.healthy,
    "blocking_dependency_conflicts": [c.message for c in DEPENDENCY_HEALTH.blocking],
    "non_blocking_dependency_conflicts": [c.message for c in DEPENDENCY_HEALTH.non_blocking],
}, indent=2, sort_keys=True))
if DEPENDENCY_HEALTH.non_blocking:
    print("NOTE: the conflicts above are outside the S1 dependency closure and are")
    print("      recorded as non-blocking diagnostics. They are NOT remediated here:")
    print("      huggingface_hub is not upgraded for Gradio, and NumPy/Torch are untouched.")


## 6. NumPy / AdamW ABI preflight

In [ ]:
# FAIL-FAST NumPy/Torch C-ABI health. This is the FIRST place NumPy or Torch is
# imported, and it runs BEFORE any Drive mount, corpus, VnCoreNLP, tokenizer, or
# model acquisition. The Audit 0022 run died here in disguise: the ABI was already
# broken, and torch.optim.AdamW merely triggered the first compiled numpy.random import.
abi_report = {
    "numpy_imported": False,
    "numpy_random_imported": False,
    "torch_imported": False,
    "adamw_constructed": False,
    "pip_check_passed": PIP_CHECK_PASSED,
}
try:
    import numpy as np  # noqa: E402
    abi_report["numpy_imported"] = True

    from numpy.random import RandomState  # noqa: E402

    rng = RandomState(42)
    values = rng.rand(4)
    assert values.shape == (4,)
    abi_report["numpy_random_imported"] = True

    import numpy.random.mtrand as numpy_mtrand  # noqa: E402

    abi_report.update({
        "numpy_version": np.__version__,
        "numpy_path": str(Path(np.__file__).resolve().parent),
        "numpy_mtrand_path": str(Path(numpy_mtrand.__file__).resolve()),
    })
    # numpy.core is a DEPRECATED NumPy 2.x compatibility shim; it is a diagnostic
    # path only, so its absence must never be read as an ABI failure.
    try:
        import numpy.core as numpy_core  # noqa: E402

        abi_report["numpy_core_path"] = str(Path(numpy_core.__file__).resolve().parent)
    except Exception as core_exc:  # noqa: BLE001 - diagnostic only
        abi_report["numpy_core_path"] = f"unavailable: {type(core_exc).__name__}"

    import torch  # noqa: E402

    abi_report["torch_imported"] = True
    abi_report.update({
        "torch_version": torch.__version__,
        "torch_path": str(Path(torch.__file__).resolve().parent),
        "cuda_available": bool(torch.cuda.is_available()),
    })

    dummy_parameter = torch.nn.Parameter(torch.zeros(1))
    dummy_optimizer = torch.optim.AdamW([dummy_parameter], lr=1e-3)
    dummy_optimizer.zero_grad(set_to_none=True)
    abi_report["adamw_constructed"] = True

    # Validate the ACTUAL S1 dependency closure: every module S1 imports must load.
    # This is the positive check that replaces a global pip check verdict.
    s1_import_failures = []
    s1_import_versions = {}
    for _distribution, _module in contract.import_closure_roots:
        try:
            _imported = importlib.import_module(_module)
            s1_import_versions[_module] = str(
                getattr(_imported, "__version__", "") or package_version(_distribution))
        except Exception as import_exc:  # noqa: BLE001 - collect every failure
            s1_import_failures.append(f"{_module}: {type(import_exc).__name__}: {import_exc}")
    abi_report["s1_import_failures"] = s1_import_failures
    abi_report["s1_import_versions"] = s1_import_versions
    abi_report["s1_dependency_closure_verified"] = True
except Exception as exc:  # noqa: BLE001 - diagnostics then fail fast
    abi_report["error"] = f"{type(exc).__name__}: {exc}"
    print(json.dumps({
        "abi_preflight": "FAILED",
        "report": abi_report,
        "python": platform.python_version(),
        "sys_path_head": sys.path[:5],
        "pythonpath": os.environ.get("PYTHONPATH", ""),
    }, indent=2, sort_keys=True))
    raise

abi_report["numpy_distribution_count"] = sum(
    1 for dist in importlib_metadata.distributions()
    if (dist.metadata["Name"] or "").lower() == "numpy")
abi_report["python_version"] = platform.python_version()
# Closure-scoped dependency health. `pip_check_output` is carried in full; only
# conflicts raised BY the S1 closure can produce a blocking problem.
abi_report.update(DEPENDENCY_HEALTH.as_dict())

abi_problems = validate_abi_report(abi_report)
assert not abi_problems, f"NumPy/Torch ABI preflight failed: {abi_problems}"
NUMPY_ABI_PREFLIGHT_PASSED = True
S1_DEPENDENCY_CLOSURE_VERIFIED = True
print(json.dumps({"abi_preflight": "PASSED", "report": abi_report}, indent=2, sort_keys=True))


## 7. Runtime, GPU, and Drive mount

In [ ]:
runtime_report = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "in_colab": IN_COLAB_BOOTSTRAP,
    "dependency_contract_version": DEPENDENCY_CONTRACT_VERSION,
    "numpy_abi_preflight_passed": NUMPY_ABI_PREFLIGHT_PASSED,
    "s1_dependency_closure_verified": S1_DEPENDENCY_CLOSURE_VERIFIED,
}
assert IN_COLAB_BOOTSTRAP, "run this evaluation on Google Colab."
assert torch.cuda.is_available(), "evaluation requires a Colab GPU runtime."
device = torch.device("cuda")
runtime_report.update({"torch": torch.__version__, "cuda_available": True,
                       "gpu_name": torch.cuda.get_device_name(0)})
importlib.import_module("google.colab.drive").mount("/content/drive")
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(json.dumps(runtime_report, indent=2, sort_keys=True))

## 8. Validate the full-training artifact (READ-ONLY gate)

Every required file, the three-way checkpoint hash agreement, run completion, epoch/step
accounting, the pinned revision, the config hash, the best metric, and the checkpoint resume
schema - including that the smoke checkpoint was **not** the initializer. Evaluation is refused
unless all of it passes. Nothing in the artifact directory is written.

In [ ]:
from mednorm_vi.training.s1_full_training import (  # noqa: E402
    BEST_METRIC_KEY,
    MentionMetrics,
    full_training_output_paths,
    is_supervised_example,
    load_full_training_config,
    validate_full_training_artifact,
)

# The pinned revision may be taken from the artifact's own manifest when the
# operator did not supply one, but it is still validated against it below.
_manifest_path = full_training_output_paths(FULL_TRAINING_ARTIFACT_DIR)["training_manifest"]
if not PINNED_MODEL_REVISION and Path(_manifest_path).is_file():
    PINNED_MODEL_REVISION = str(json.loads(
        Path(_manifest_path).read_text(encoding="utf-8"))["model"]["pinned_model_revision"])
config = load_full_training_config(
    REPO_DIR / "configs" / "training" / "s1_mention_full_training.yaml",
    pinned_revision=PINNED_MODEL_REVISION)

from mednorm_vi.training.s1_full_training import (  # noqa: E402
    resolve_s1_best_checkpoint,
    validate_best_checkpoint_only,
)

CHECKPOINT_LOCATION = resolve_s1_best_checkpoint(
    repository_root=REPO_DIR, in_colab=IN_COLAB_BOOTSTRAP)
print(json.dumps(CHECKPOINT_LOCATION.as_dict(), indent=2, sort_keys=True))

ARTIFACT_PATHS = full_training_output_paths(FULL_TRAINING_ARTIFACT_DIR)
checkpoint_metadata = {}
_candidates = {"best_checkpoint": CHECKPOINT_LOCATION.path}
if EXECUTION_MODE == "colab":
    _candidates["latest_checkpoint"] = Path(ARTIFACT_PATHS["latest_checkpoint"])
for _name, _path in _candidates.items():
    if _path.is_file():
        _loaded = torch.load(_path, map_location="cpu", weights_only=False)  # read-only
        checkpoint_metadata[_name] = {
            k: v for k, v in _loaded.items() if not k.endswith("_state_dict")}
        checkpoint_metadata[_name].update(
            {k: True for k in _loaded if k.endswith("_state_dict")})
        if _name == "best_checkpoint":
            best_state_dict = _loaded["model_state_dict"]
        del _loaded

if EXECUTION_MODE == "colab":
    # COLAB: the complete Drive artifact is present, so the Audit 0031
    # full-artifact validator runs unchanged and unweakened.
    artifact_outcome = validate_full_training_artifact(
        FULL_TRAINING_ARTIFACT_DIR,
        expected_checkpoint_sha256={
            "best_checkpoint": EXPECTED_BEST_CHECKPOINT_SHA256,
            "latest_checkpoint": EXPECTED_LATEST_CHECKPOINT_SHA256,
        },
        expected_pinned_revision=PINNED_MODEL_REVISION,
        checkpoint_payloads=checkpoint_metadata,
    )
    FULL_ARTIFACT_VALIDATED = artifact_outcome.validated
    BEST_CHECKPOINT_VALIDATED = artifact_outcome.validated
    gate_report = artifact_outcome.as_dict()
    failures = artifact_outcome.failures
    scope = "full_training_artifact"
else:
    # LOCAL: only best.pt exists, so the narrower best-checkpoint-only gate runs.
    # It reports full_artifact_validated=False, which is carried into the report;
    # the full artifact must still be validated on Drive.
    artifact_outcome = validate_best_checkpoint_only(
        CHECKPOINT_LOCATION.require(),
        expected_sha256=EXPECTED_BEST_CHECKPOINT_SHA256,
        expected_pinned_revision=PINNED_MODEL_REVISION,
        expected_epoch=EXPECTED_COMPLETED_EPOCHS,
        expected_global_step=EXPECTED_COMPLETED_OPTIMIZER_STEPS,
        payload=checkpoint_metadata.get("best_checkpoint"),
    )
    FULL_ARTIFACT_VALIDATED = False
    BEST_CHECKPOINT_VALIDATED = artifact_outcome.best_checkpoint_validated
    gate_report = artifact_outcome.as_dict()
    failures = artifact_outcome.failures
    scope = "best_checkpoint_only"

print(json.dumps(gate_report, indent=2, sort_keys=True))
if failures:
    for _failure in failures:
        print("  -", _failure)
    raise AssertionError(
        f"{scope} validation failed ({len(failures)} condition(s)); "
        "evaluation is not authorized")
print(f"{scope.upper()} VALIDATED - evaluation authorized "
      f"(full_artifact_validated: {FULL_ARTIFACT_VALIDATED}).")

## 9. Governed corpus gate and registry

In [ ]:
from mednorm_vi.model_registry.registry import load_registry, validate_profile_budget
from mednorm_vi.training.phobert_alignment import (
    resolve_segmented_text,
    describe_backend,
    map_segmented_words,
    segmented_text_to_words,
    verify_tokenizer_equivalence,
)
from mednorm_vi.training.s1_mention_smoke import (
    ENTITY_TYPE_ORDER,
    run_alignment_preflight,
    encode_mention_example_slow,
    load_governed_exclusions,
    expected_corpus_from_config,
    iter_jsonl,
    load_coverage,
    loss_mask_for_example,
    pad_encoded_features,
    sha256_file,
    verify_governed_corpus,
)

expected_corpus = expected_corpus_from_config(config.raw)
corpus_report = verify_governed_corpus(CORPUS_DIR, expected_corpus)
coverage_by_source = load_coverage(CORPUS_DIR)

roles = load_registry(REPO_DIR / "configs" / "model_registry" / "models_v1.yaml")
role = next(r for r in roles if r.model_id == config.registry_model_id)
profile_budget = validate_profile_budget(roles, profile="full")
assert profile_budget.within_9b, "full model profile exceeds the 9B budget"
print(json.dumps({**corpus_report, "registry_role": role.role,
                  "full_profile_within_9b": profile_budget.within_9b},
                 indent=2, sort_keys=True))

## 10. Word segmentation contract (VnCoreNLP RDRSegmenter required)

In [ ]:
VNCORENLP_DIR = DRIVE_ROOT / "model_cache" / "vncorenlp"
# Production S1 training REQUIRES VnCoreNLP RDRSegmenter. whitespace_fallback is an
# explicit opt-in diagnostic mode only; it never activates automatically.
SEGMENTER_MODE = os.environ.get("MEDNORM_SEGMENTER_MODE", "vncorenlp")
assert SEGMENTER_MODE in ("vncorenlp", "whitespace_fallback"), SEGMENTER_MODE
DEGRADED_FALLBACK = SEGMENTER_MODE == "whitespace_fallback"

segmenter_report = {
    "segmenter_mode": SEGMENTER_MODE,
    "word_segmenter": "",
    "word_segmenter_version": "",
    "word_segmenter_resource_hashes": {},
    "degraded_fallback": DEGRADED_FALLBACK,
    "resource_dir": str(VNCORENLP_DIR),
    "acquisition_source": "",
}

if SEGMENTER_MODE == "vncorenlp":
    # py_vncorenlp was installed in the single constrained transaction (PASS 1).
    VNCORENLP_DIR.mkdir(parents=True, exist_ok=True)
    py_vncorenlp = importlib.import_module("py_vncorenlp")
    if not any(VNCORENLP_DIR.glob("*.jar")):
        py_vncorenlp.download_model(save_dir=str(VNCORENLP_DIR))
    _resources = sorted(
        [q for q in VNCORENLP_DIR.rglob("*") if q.is_file()], key=lambda q: q.name)
    assert _resources, "VnCoreNLP resources missing after acquisition (fail fast)"
    _hashes = {q.name: sha256_file(q) for q in _resources if q.suffix in (".jar", ".xz", ".txt")}
    assert _hashes, "VnCoreNLP resource hashes are empty (broken installation; fail fast)"
    _rdr = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=str(VNCORENLP_DIR))
    segmenter_report.update({
        "word_segmenter": "VnCoreNLP RDRSegmenter",
        "word_segmenter_version": "py_vncorenlp==0.1.4",
        "word_segmenter_resource_hashes": _hashes,
        "acquisition_source": "py_vncorenlp.download_model",
    })

    def _run_segmenter(text):
        segments = _rdr.word_segment(text)
        assert segments, "RDRSegmenter returned no segments (fail fast)"
        return " ".join(segments)
else:
    print("=" * 78)
    print("!! DEGRADED MODE: whitespace_fallback is NOT the production S1 path.")
    print("!! Word segmentation does not match ViHealthBERT-Word pretraining.")
    print("!! This run cannot be classified as a successful production-path S1 smoke.")
    print("=" * 78)
    segmenter_report.update({
        "word_segmenter": "whitespace (degraded diagnostics only)",
        "word_segmenter_version": "builtin-whitespace",
        "acquisition_source": "none (degraded diagnostics mode)",
    })

    def _run_segmenter(text):
        return " ".join(text.split())


# SEGMENTATION POLICY (single rule for raw and pre-segmented sources):
# text that is ALREADY RDRSegmenter output is used verbatim - re-segmenting it
# splits the join character off as a standalone token and shreds the very words
# ViHealthBERT-Word expects. Everything else goes through the production segmenter.
def segment_example_text(text):
    segmented, _source = resolve_segmented_text(text, _run_segmenter)
    return segmented


segmenter_report["segmentation_policy"] = (
    "pre_segmented_source_used_verbatim_else_vncorenlp")

PRODUCTION_SEGMENTATION = (
    segmenter_report["segmenter_mode"] == "vncorenlp"
    and segmenter_report["word_segmenter"] == "VnCoreNLP RDRSegmenter"
    and segmenter_report["degraded_fallback"] is False
    and bool(segmenter_report["word_segmenter_resource_hashes"])
)
print(json.dumps({k: v for k, v in segmenter_report.items()
                  if k != "word_segmenter_resource_hashes"}, indent=2, sort_keys=True))
print("resource_hash_count", len(segmenter_report["word_segmenter_resource_hashes"]))
print("production_segmentation", PRODUCTION_SEGMENTATION)


In [ ]:
assert PRODUCTION_SEGMENTATION, (
    "evaluation requires production VnCoreNLP segmentation; "
    "whitespace_fallback is diagnostics only")

## 11. Slow tokenizer at the pinned revision

In [ ]:
os.environ["HF_HOME"] = str(MODEL_CACHE_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

transformers_module = importlib.import_module("transformers")
AutoModel = transformers_module.AutoModel
AutoTokenizer = transformers_module.AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    config.hf_model_id, revision=PINNED_MODEL_REVISION,
    cache_dir=str(MODEL_CACHE_DIR), use_fast=False)
tokenizer_report = describe_backend(tokenizer)
assert tokenizer_report["tokenizer_is_fast"] is False
PAD_TOKEN_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1
print(json.dumps(tokenizer_report, indent=2, sort_keys=True))

## 12. Alignment preflight over the governed internal-test split

The same segmentation, character alignment, mention encoding and tokenizer-equivalence
contracts the training run used. Zero unexpected unalignable examples are required.

In [ ]:
GOVERNED_EXCLUSIONS = load_governed_exclusions(
    REPO_DIR / "configs" / "training" / "s1_governed_exclusions.yaml")


def segment_for_alignment(text):
    segmented, _source = resolve_segmented_text(text, _run_segmenter)
    return segmented


def encode_one(row):
    return encode_mention_example_slow(
        row, tokenizer, coverage_by_source=coverage_by_source,
        max_length=config.max_sequence_length,
        segmented_text=segment_for_alignment(row["text"]))


def verify_one(row):
    words = map_segmented_words(
        row["text"], segmented_text_to_words(segment_for_alignment(row["text"])))
    verify_tokenizer_equivalence(words, tokenizer)


def supervised_row(row):
    return is_supervised_example(loss_mask_for_example(row, coverage_by_source))


eval_preflight, eval_features = run_alignment_preflight(
    {EVAL_SPLIT: iter_jsonl(CORPUS_DIR / "splits" / f"{EVAL_SPLIT}.jsonl")},
    encode=encode_one, verify_equivalence=verify_one,
    governed_exclusions=GOVERNED_EXCLUSIONS, supervised=supervised_row)
eval_preflight_report = eval_preflight.as_dict()
print(json.dumps({k: v for k, v in eval_preflight_report.items()
                  if k not in ("unalignable_examples", "governed_exclusions")},
                 indent=2, sort_keys=True))
assert eval_preflight.passed, (
    f"internal-test alignment preflight failed: "
    f"{eval_preflight.unalignable_example_count} unexpected unalignable example(s)")
internal_test_features = eval_features[EVAL_SPLIT]
assert internal_test_features, "no evaluable internal-test features"

## 13. Load the best checkpoint into the trained architecture

The backbone comes from the **pinned pretrained revision** and the trained weights come from
`best.pt`. No optimizer or scheduler is constructed and no gradient is ever enabled.

In [ ]:
class MentionTokenClassifier(torch.nn.Module):
    def __init__(self, base_model: torch.nn.Module, label_count: int) -> None:
        super().__init__()
        self.base_model = base_model
        hidden_size = int(base_model.config.hidden_size)
        self.dropout = torch.nn.Dropout(0.1)
        self.classifier = torch.nn.Linear(hidden_size, label_count)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        output = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(self.dropout(output.last_hidden_state))


# The checkpoint holds the COMPLETE state dict, so the architecture can be built
# from the cached config alone. That avoids downloading base weights locally while
# giving a bit-identical model after the strict load below.
try:
    backbone = AutoModel.from_pretrained(
        config.hf_model_id, revision=PINNED_MODEL_REVISION, cache_dir=str(MODEL_CACHE_DIR))
    BACKBONE_SOURCE = "pretrained_weights"
except Exception:  # noqa: BLE001 - fall back to config-only reconstruction
    _model_config = transformers_module.AutoConfig.from_pretrained(
        config.hf_model_id, revision=PINNED_MODEL_REVISION, cache_dir=str(MODEL_CACHE_DIR))
    backbone = AutoModel.from_config(_model_config)
    BACKBONE_SOURCE = "config_only_then_strict_state_dict"
model = MentionTokenClassifier(backbone, len(ENTITY_TYPE_ORDER)).to(device)
missing, unexpected = model.load_state_dict(best_state_dict, strict=True), None
model.eval()
for _parameter in model.parameters():
    _parameter.requires_grad_(False)
print(json.dumps({
    "loaded_from": str(CHECKPOINT_LOCATION.path),
    "execution_mode": EXECUTION_MODE,
    "backbone_source": BACKBONE_SOURCE,
    "pinned_model_revision": PINNED_MODEL_REVISION,
    "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
    "strict_state_dict_load": True,
}, indent=2, sort_keys=True))

## 14. Evaluate on internal_test (forward passes only)

In [ ]:
def collate(batch):
    padded = pad_encoded_features(batch, pad_token_id=PAD_TOKEN_ID)
    return {
        "input_ids": torch.tensor(padded["input_ids"], dtype=torch.long),
        "attention_mask": torch.tensor(padded["attention_mask"], dtype=torch.long),
        "labels": torch.tensor(padded["labels"], dtype=torch.float32),
        "label_mask": torch.tensor(padded["label_mask"], dtype=torch.float32),
    }


loader = torch.utils.data.DataLoader(
    internal_test_features, batch_size=config.per_device_batch_size,
    shuffle=False, collate_fn=collate, num_workers=0)

metrics = MentionMetrics()
started = time.time()
with torch.no_grad():
    for batch in loader:
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        logits = model(batch["input_ids"], batch["attention_mask"])
        predictions = (torch.sigmoid(logits.float()) > config.decision_threshold).int()
        metrics.update(predictions.tolist(), batch["labels"].int().tolist(),
                       batch["label_mask"].int().tolist())
internal_test_metrics = metrics.compute()
internal_test_metrics["evaluated_examples"] = len(internal_test_features)
internal_test_metrics["evaluation_seconds"] = round(time.time() - started, 1)
print(json.dumps(internal_test_metrics, indent=2, sort_keys=True))

## 15. Evaluation report

Written to the **separate** evaluation directory. The full-training artifact is untouched.

In [ ]:
evaluation_report = {
    "report_version": 1,
    "stage_id": "S1",
    "mode": "READ_ONLY_EVALUATION",
    "split": EVAL_SPLIT,
    "decision_threshold": config.decision_threshold,
    "best_metric_key": BEST_METRIC_KEY,
    "internal_test_metrics": internal_test_metrics,
    "validation_best_metric_from_training": getattr(artifact_outcome, "best_metric", None),
    "execution_mode": EXECUTION_MODE,
    "full_artifact_validated": FULL_ARTIFACT_VALIDATED,
    "best_checkpoint_validated": BEST_CHECKPOINT_VALIDATED,
    "validation_scope": scope,
    "full_training_artifact": {
        "artifact_dir": str(FULL_TRAINING_ARTIFACT_DIR),
        "validated": FULL_ARTIFACT_VALIDATED,
        "checkpoint_sha256": artifact_outcome.checkpoint_sha256,
        "pinned_model_revision": artifact_outcome.pinned_model_revision,
    },
    "corpus": corpus_report,
    "alignment_preflight": eval_preflight_report,
    "tokenizer": tokenizer_report,
    "word_segmentation": segmenter_report,
    "runtime": runtime_report,
    "organizer_inference_run": False,
    "output_zip_generated": False,
    "training_artifact_modified": False,
}
report_path = EVAL_OUTPUT_DIR / "internal_test_evaluation.json"
report_path.write_text(json.dumps(evaluation_report, indent=2, sort_keys=True),
                       encoding="utf-8")
print(json.dumps({
    "report_path": str(report_path),
    "internal_test_span_micro_f1": internal_test_metrics[BEST_METRIC_KEY],
    "internal_test_token_micro_f1": internal_test_metrics["token_micro_f1"],
    "evaluated_examples": internal_test_metrics["evaluated_examples"],
}, indent=2, sort_keys=True))

## 16. Return artifacts

Return `s1_mention_internal_test_eval_v1/internal_test_evaluation.json` for review.

The full-training artifact was read only. No organizer inference was run, no `output.zip` was
produced, and no model weights were copied into the repository.